In [1]:
# Forzar que la carpeta raíz del repo esté en sys.path para imports locales
import sys, os
repo_root = os.path.abspath('..')  # notebook está en experiments/
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Repo root added to sys.path:', repo_root)


Repo root added to sys.path: c:\Users\marco\Documents\Deusto\procesamiento_del_lenguaje_natural\fallacy-classification


In [2]:
# Importar utilidades y librerías
from experiments.utils import load_and_prepare_datasets, build_vectorizers, encode_labels, evaluate_classification
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
import pandas as pd

c:\Users\marco\Documents\Deusto\procesamiento_del_lenguaje_natural\fallacy-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cargar dataset combinado (limitado para pruebas rápidas)
train_texts, train_labels, dev_texts, dev_labels, test_texts, test_labels = load_and_prepare_datasets(limit_per_source=2000)
print(f'Loaded: train={len(train_texts)}, dev={len(dev_texts)}, test={len(test_texts)}')

Filter: 100%|██████████| 1776/1776 [00:00<00:00, 54896.60 examples/s]

Loaded: train=2000, dev=400, test=400


In [4]:
# Construir representaciones BoW y TF-IDF
bow, tfidf = build_vectorizers(train_texts)
X_train_bow = bow.transform(train_texts)
X_dev_bow = bow.transform(dev_texts)
X_test_bow = bow.transform(test_texts)

X_train_tfidf = tfidf.transform(train_texts)
X_dev_tfidf = tfidf.transform(dev_texts)
X_test_tfidf = tfidf.transform(test_texts)

y_train, y_dev, y_test, le = encode_labels(train_labels, dev_labels, test_labels)

In [5]:
# Entrenar Logistic Regression sobre BoW (grid pequeño)
lr = LogisticRegression(max_iter=1000)
grid = GridSearchCV(lr, {"C": [0.1, 1.0, 10.0]}, cv=3, n_jobs=1)
grid.fit(X_train_bow, y_train)
pred = grid.predict(X_dev_bow)
metrics = evaluate_classification(y_dev, pred)
metrics.update({"model": "LogisticRegression", "representation": "BoW", "best_params": str(grid.best_params_)})
print(metrics)

{'accuracy': 0.4275, 'precision_macro': 0.4635822687353596, 'recall_macro': 0.41276536212005, 'f1_macro': 0.41692944369388657, 'model': 'LogisticRegression', 'representation': 'BoW', 'best_params': "{'C': 10.0}"}


In [6]:
# Entrenar LinearSVC sobre TF-IDF (grid pequeño)
svc = LinearSVC(max_iter=5000)
grid2 = GridSearchCV(svc, {"C": [0.1, 1.0, 10.0]}, cv=3, n_jobs=1)
grid2.fit(X_train_tfidf, y_train)
pred2 = grid2.predict(X_dev_tfidf)
metrics2 = evaluate_classification(y_dev, pred2)
metrics2.update({"model": "LinearSVC", "representation": "TF-IDF", "best_params": str(grid2.best_params_)})
print(metrics2)

{'accuracy': 0.455, 'precision_macro': 0.48456232835265084, 'recall_macro': 0.4309752986220455, 'f1_macro': 0.4366230793019818, 'model': 'LinearSVC', 'representation': 'TF-IDF', 'best_params': "{'C': 1.0}"}
